## Problem framing

**Problem type:** binary classification

**Target:** `Exited`
- 1 (positive class) = churn: the customer leaves the bank
- 0 (negative class) = the customer stays

The target is imbalanced: 21% churn (see 01_eda).

**Business objective:** run a targeted retention campaign. To do so, the bank needs to predict which customers are likely to leave.

**Features:**
- **ID**: row identifier
- **CustomerId**: unique customer identifier : <span style="color: red;">stated as unique, but not the case (see 01_eda)</span>
- **Surname**: customer's surname
- **CreditScore**: numerical value representing the customer's credit score
- **Geography**: country of residence (France, Spain or Germany)
- **Gender**: customer's gender (Male or Female)
- **Age**: customer's age
- **Tenure**: number of years the customer has been with the bank
- **Balance**: account balance
- **NumOfProducts**: number of bank products held (e.g. savings account, credit card)
- **HasCrCard**: whether the customer has a credit card (1 = yes, 0 = no)
- **IsActiveMember**: whether the customer is an active member (1 = yes, 0 = no)
- **EstimatedSalary**: customer's estimated salary

## Choice of the main metric

### 1. Cost of errors
- Positive class: 1 = churn.
- The most costly error is the **false negative**: a customer who leaves without being detected. I therefore primarily aim to maximize **recall**.
- **False positives** also have a cost: every retention offer sent to a customer who would have stayed wastes campaign budget. I therefore also monitor **precision**.
- ⚠️ <span style="color: red;"> Assumption to confirm: "a FN costs more than a FP" is not quantified (value of a retained customer vs. cost of an offer). This is a red flag.</span>

**Formulas:**
- Recall (TPR) = TP / (TP + FN)
- FPR = FP / N = FP / (FP + TN)
- Precision = TP / (TP + FP)
- Accuracy = (TP + TN) / (P + N)

### 2. Naive model test
Extreme cases on 100 customers: 21 churners, 79 who stay.
A metric is **fooled** when it gives a good score to a useless model.

**Naive model 1: always predicts churn = 1**
- Recall = 21 / (21 + 0) = 1 → **fooled**: recall ignores FPs
- FPR = 79 / 79 = 1
- Precision = 21 / (21 + 79) = 0.21
- Accuracy = (21 + 0) / 100 = 0.21

→ FPR, precision and accuracy expose the problem.

**Naive model 2: always predicts churn = 0**
- Recall = 0 / (0 + 21) = 0
- FPR = 0 / 79 = 0 → **fooled**: FPR ignores FNs
- Precision = 0 / (0 + 0) → undefined (sklearn returns 0 with a warning), so not fooled
- Accuracy = (0 + 79) / 100 = 0.79 → **fooled**: accuracy does not distinguish FNs from FPs and is dominated by the majority class

→ Only recall exposes the problem.

### 3. Choice of the metric pair
- **Recall** is required: it is the only metric that exposes naive model 2.
- **FPR** and **accuracy** are ruled out: they are fooled by naive model 2.
- **Precision** is not fooled, so it complements recall. It also answers the business question: "among targeted customers, how many actually leave?"

→ Selected pair: **recall + precision**, combined into **F1 = 2 × P × R / (P + R)**.

### 4. Decision threshold
- F1 is computed for a given threshold (0.5 by default).
- Lowering the threshold makes the model predict more churners: recall goes up and precision goes down.
- The **Precision-Recall curve** shows this trade-off across all thresholds. **PR-AUC** is the area under this curve. The selected threshold is the one that **maximizes F1**.
- Edge case: a PR-AUC of 1 means the classes are perfectly separated, and some threshold gives precision = recall = 1. This is extremely rare in practice.
- The threshold is chosen on the **training set, using cross-validation**, never on the test set. Otherwise the test score is no longer an unbiased estimate: it is overly optimistic.

### 5. Conclusion
- **Main metric: F1**, set by the Kaggle competition and consistent with the reasoning above.
- **Secondary metrics**: recall and precision, tracked separately. The business favors recall, whereas F1 weighs both equally. In a real project, an F-beta score (β > 1) would be used, with β set from the FN/FP cost ratio, currently unknown (red flag).

## Fundamental analysis

For each question: **Observed** = seen in the data (with the `01_eda` section as evidence), **Documented** = stated by a source (with the link), **Assumed** = hypothesis, not verified.
Status: **Known** / **Partial** / **Unknown**. Every Unknown or Partial answer feeds the red flags.

### Feature review (meaning and measurement time)

Source of definitions: competition data description. No documentation on how or when each feature was measured.

| Feature | Definition documented? | Leakage risk before EDA (hypothesis) | Leakage risk after EDA | Remark |
|---|---|---|---|---|
| `CreditScore` | Partial: no provider, no calculation method | Low | Low (negligible link with churn) | Observed range 350–850, consistent with common credit score scales |
| `Geography` | Yes | Low | Low | Only France, Spain, Germany |
| `Gender` | Yes | Low | Low | Ethical question: should a targeting model treat customers differently by gender? |
| `Age` | Yes | Low | Low | Measurement date unknown |
| `Tenure` | Partial: unit (full years?) and reference date not stated | Low to medium | Low (no trend with churn) | If measured at churn date, it reflects how long the customer stayed |
| `Balance` | Partial: balance at which date? | **Possible** | **Low**: zero balances churn *less*, the opposite of the leakage scenario (01_eda §13) | If measured after the account was closed, it could be 0 because of the churn itself |
| `NumOfProducts` | Partial: at which date? | **Possible** | Possible **at 1 product only** (gradual closing); implausible at 3–4 (01_eda §9) | A leaving customer may close products before closing the account |
| `HasCrCard` | Yes | Possible | Negligible (2-point gap, 01_eda §10) | Same as NumOfProducts |
| `IsActiveMember` | No: "active" is not defined (activity type, period) | **Possible** | **Main risk**: inactivity can follow churn (01_eda §11) | A customer becomes inactive before or after leaving: the timing matters |
| `EstimatedSalary` | Partial: estimated how, by whom? | Low | Low | An estimate, not a declared salary |

The leakage risks before EDA were hypotheses. The EDA found no global leakage (no variable separates churners almost perfectly), but the risks cannot be ruled out without the measurement dates.

### The 6 questions

| Question | Observed (01_eda) | Documented (source) | Assumed | Status |
|---|---|---|---|---|
| **What**: what does a row represent? what exactly is `Exited`? | Labelled file: 143,579 rows. `ID` is unique, but only 22,017 unique `CustomerId` and 2,765 unique `Surname`; the same `CustomerId` and the same `Surname` are shared by unrelated people (different gender, country, age, tenure) (01_eda §5, §6). `Exited` is binary, imbalanced: 21% churn (01_eda §3). `ID` matches the S4E1 `id` exactly (01_eda §4). | Competition description: each row is a bank customer; `Exited` = whether the customer left the bank (1 = yes, 0 = no). S4E1: rows were generated by a deep learning model (01_eda §4). | The generator produced `CustomerId` and `Surname` like any other column, without linking them to a single person, and partially preserved the surname–country link of the original data (consistent with 01_eda §5, §6; generator details not documented). | Partial: each row is one customer profile, but with no reliable identifier, we cannot check whether several rows describe the same customer (row independence not verifiable). The churn time window and scope (all products or one) are not documented. |
| **Who**: who produced the data? which customers? | Customers from France, Spain and Germany, aged 18 to 92, both genders (01_eda §2). | This version: private Kaggle competition of the ML-PRO training program. Kaggle Playground S4E1: data generated by Kaggle from a model trained on the "Bank Customer Churn Prediction" dataset. | Chain of producers: an unknown bank (original data) → Kaggle (synthetic S4E1 data) → ML-PRO (labelled file + test set without target; labelled file verified row by row, see How). | Partial: the customer population is described and the producers of the synthetic data are identified, but the bank behind the original data is unknown. |
| **When**: when were features and churn measured? features before churn? | No date column in the data. | Nothing: no period and no measurement date are given. | — | Unknown: we cannot tell whether the features (e.g. `Balance`, `NumOfProducts`, `IsActiveMember`) were measured before the churn, nor whether the data is recent. |
| **Where**: which bank, which countries? | `Geography` has 3 values: France (57%), Spain (22%), Germany (21%) (01_eda §6). | Competition description: `Geography` = country of residence (France, Spain or Germany). The bank is not named. | A European or international bank (not documented). | Partial: countries known, bank unknown. |
| **How**: how was the data collected or generated? | Our labelled file + ML-PRO test set = 143,579 + 21,455 = 165,034 rows, `ID` from 0 to 165,033 (01_eda §4). S4E1 train: 165,034 rows, `id` from 0 to 165,033, same columns, dtypes and ranges, slightly more unique values than our labelled file. **Row-by-row check: all 143,579 rows of our labelled file are found in S4E1 with 0 difference in every column** (01_eda §4). The same `CustomerId` and `Surname` are shared by unrelated people (01_eda §5, §6). Some surnames contain "?" (encoding issue). `Surname` is not independent of `Geography`: χ² p < 0.001, Cramér's V ≈ 0.19 (01_eda §6). | Our competition page: nothing. [Kaggle Playground S4E1](https://www.kaggle.com/competitions/playground-series-s4e1/data) (train: 165,034 rows): "generated from a deep learning model trained on the Bank Customer Churn Prediction dataset. Feature distributions are close to, but not exactly the same, as the original." | The ML-PRO test set (21,455 rows) contains the remaining S4E1 rows (same total; not checked row by row). The generation explains the identifiers and surnames shared by unrelated people and the partial surname–country link. | Partial: our labelled file is an exact subset of S4E1, whose data is documented as generated by a deep learning model; the generator details (model, how identifiers were assigned) are not documented. |
| **Why**: why was the dataset built? | — | This version: private Kaggle competition of the ML-PRO training program, to predict churn. | The original data was extracted by the bank to study churn (not verifiable). | Partial: the purpose of this version is known (training exercise; S4E1 is a Kaggle practice competition); the purpose and context of the original data collection are not. |

### Red flags
- **`CustomerId` is not a real identifier**: shared by unrelated people. `CustomerId`, `Surname` and `ID` are not used as features.
- **Original data producer unverifiable**: the bank behind the original data is unknown. The synthetic data was produced by Kaggle (S4E1) and reused by ML-PRO (verified for the labelled file).
- **Row independence not verifiable**: the rows are synthetic samples from a generator trained on the original customers (dataset size not documented), so several rows may be close variants of the same original customer. If such rows end up in both train and test, scores would be optimistic. Not verifiable without the original data and the generator details.
- **Geographic scope**: only France, Spain and Germany; the bank is unknown. The model is not applicable to other countries. France dominates (57%): performance should be checked per country at evaluation.
- **Synthetic data**: our labelled file is an exact subset of S4E1, generated by a deep learning model. Relationships between features and `Exited` may reflect the generator rather than real customer behavior, and distributions differ slightly from the original data. Some features look like random draws rather than measurements: `EstimatedSalary` has more high salaries than low ones, and `Tenure` is almost uniform with no link to churn (01_eda §12, §16).
- **Character encoding issues**: some surnames contain "?": non-ASCII characters were lost, which signals limited quality control on the data pipeline.
- **When unknown (possible leakage)**: if some features were measured after (or because of) the churn, e.g. `Balance` at 0 after closing the account, the model would learn the consequence instead of the cause. To be watched in the EDA.
- **FN/FP costs not quantified**: the metric choice relies on the assumption that a FN costs more than a FP (see the metric section).
- **Undocumented feature definitions**: `IsActiveMember` (what is "active"?), `CreditScore` (provider, method), `EstimatedSalary` (estimation method), `Tenure` (full years? reference date?).
- **Ethical risks**: `Gender` in a targeting model may lead to unequal treatment of customers by gender (to be discussed at feature selection). `Surname` could act as a proxy for ethnic origin (already excluded).
- **Test labels publicly available**: the ML-PRO test set (21,455 rows) most likely comes from the S4E1 train set, which includes `Exited`. S4E1 must never be used for training, tuning or evaluating the model (leakage). It is only used to verify the data origin (01_eda §4).
- **Germany: 99.7% positive balances** (vs about 31% in France and Spain, 01_eda §18.1). Either an artifact of how the original data was collected, or a real pattern tied to this bank (e.g. German customers using it as their main bank). Not verifiable; in both cases the higher German churn may not generalize to another bank.
- **Target incompletely defined**: the time window of the churn and its scope (all products or only one) are not documented.
- **Applicability (summary)**: the model can only be claimed valid for bank customers in France, Spain and Germany, on data produced the same way as this dataset (origin and generation method unknown), and for `Exited` as defined here (no known time window or scope). Its performance on real customers of a real bank cannot be guaranteed.

### Open questions (to check in 01_eda)
| Question | Where | Status |
|---|---|---|
| Leakage: do `Balance`, `NumOfProducts`, `HasCrCard` or `IsActiveMember` separate churners almost perfectly (e.g. `Balance` = 0 for nearly all churners)? | 01_eda | answered: no global leakage. `Balance` = 0 is linked to **lower** churn (opposite of the leakage scenario); `HasCrCard` negligible; `NumOfProducts` possible timing effect at 1 product; `IsActiveMember` is the most serious timing risk (inactivity can follow churn). Measurement dates remain unknown (01_eda §9–11, §13). |
| Is the churn rate (and later the model performance) different across France, Spain and Germany? | 01_eda, then evaluation | answered for churn rate: France 16.5%, Spain 17.2%, Germany 38.1%; χ² p < 0.001, Cramér's V = 0.21 (01_eda §7). Performance per country: to check at evaluation. |
| `Gender`: keep it as a feature or not (ethical risk)? | feature selection | open |